[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mahsa-agz/mahsa-agz.github.io/blob/main/ds-handbook/exam_prep/day_06/4_ai.ipynb)

# Day 6: AI and ML (easy)

**How to work:** Answer concept questions out loud in 60 to 90 seconds before opening the solution: definition, why it matters, one example, one trade-off.

**Today (25 min):**

| # | Question | Level | Minutes |
|---|---|---|---|
| Q1 | L1 or L2? | easy | 4 |
| Q2 | Watch coefficients disappear | easy | 6 |
| Q3 | Eighty percent on random labels | easy | 7 |
| Q4 | Pick C without touching the test set | easy | 5 |
| Q5 | Turn the penalty knob (review) | easy | 3 |

**Handbook:** [AI and ML](https://mahsa-agz.github.io/ds-handbook/#ai), [cheat sheets](https://mahsa-agz.github.io/ds-handbook/#cheat)

## Data

- Built-in scikit-learn data (no download): `load_diabetes()` (442 patients, 10 features, regression) and `load_breast_cancer()` (569 tumours, 30 features, 1 = benign). Q3 uses simulated noise on purpose (said in the question).

The setup cell loads every table into a pandas DataFrame (from your Drive folder, or downloads it). Everything runs on the free Colab CPU in seconds. Numbers in the solutions come from scikit-learn 1.8; other versions can differ in the last digit.

## Setup

Run this cell once before the questions.

In [ ]:
# Finds the data folder (local copy of the repository, or /content/data in Colab) and downloads a missing file
# from the ds-handbook repository on GitHub (or from its original source).
import os, sys, urllib.request

RAW = "https://raw.githubusercontent.com/mahsa-agz/mahsa-agz.github.io/main/ds-handbook/exam_prep/data/"

def _find_data():
    if "google.colab" in sys.modules:
        os.makedirs("/content/data", exist_ok=True)
        return "/content/data"
    for p in ("../data", "data", "exam_prep/data"):
        if os.path.isdir(p):
            return p
    os.makedirs("data", exist_ok=True)
    return "data"

DATA = _find_data()
print("data folder:", DATA)

def data_path(name, url=None):
    """Local path of a data file; downloads it from the repository (or from url) if it is not there yet."""
    path = os.path.join(DATA, name)
    if not os.path.exists(path):
        for source in (RAW + name, url):
            if not source:
                continue
            try:
                print("downloading", name)
                urllib.request.urlretrieve(source, path)
                break
            except Exception as e:
                print("  could not download from", source, "->", e)
    return path

import io, zipfile, warnings
import numpy as np
import pandas as pd
warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=DeprecationWarning)
pd.set_option("display.width", 120)

SEABORN = "https://raw.githubusercontent.com/mwaskom/seaborn-data/master/"
TELCO_URL = "https://raw.githubusercontent.com/IBM/telco-customer-churn-on-icp4d/master/data/Telco-Customer-Churn.csv"
BANK_URL = "https://archive.ics.uci.edu/static/public/222/bank+marketing.zip"

def load_telco():
    """IBM Telco churn, 7,043 customers. Target: Churn (Yes/No)."""
    return pd.read_csv(data_path("telco_churn.csv", TELCO_URL))

def load_seaborn(name):
    """titanic, penguins, diamonds, mpg, tips ... from the seaborn-data repo."""
    return pd.read_csv(data_path(name + ".csv", SEABORN + name + ".csv"))

def load_bank():
    """UCI Bank Marketing (bank-additional-full), 41,188 calls. Target: y (yes/no). The source is a zip in a zip."""
    path = data_path("bank_marketing.csv")
    if not os.path.exists(path):
        print("downloading bank_marketing (zip)")
        outer = zipfile.ZipFile(io.BytesIO(urllib.request.urlopen(BANK_URL).read()))
        inner = zipfile.ZipFile(io.BytesIO(outer.read("bank-additional.zip")))
        with open(path, "wb") as f:
            f.write(inner.read("bank-additional/bank-additional-full.csv"))
    return pd.read_csv(path, sep=";")

def check(fn, cases, tol=1e-6):
    """Runs fn on each (args, expected) case; numbers and arrays are compared with a tolerance."""
    ok = 0
    for args, want in cases:
        args = args if isinstance(args, tuple) else (args,)
        try:
            got = fn(*args)
            good = got is not None and np.shape(got) == np.shape(want) and bool(
                np.allclose(np.asarray(got, dtype=float), np.asarray(want, dtype=float), atol=tol))
        except Exception as e:
            got, good = "error: " + repr(e)[:120], False
        ok += good
        shown = " ".join(repr(got).split())
        shown = shown if len(shown) < 100 else shown[:97] + "..."
        print(("PASS " if good else "FAIL ") + f"{fn.__name__}(...) -> {shown}" + ("" if good else f"   expected {want!r}"))
    print(f"{ok}/{len(cases)} passed")

---
### Q1. L1 or L2?

*easy, about 4 min*

Compare L1 (lasso) and L2 (ridge) regularization in about 90 seconds: the penalised loss, what each does to the coefficients and why, when you would pick each, and what elastic net adds.

*Your answer (double-click to write):*

<details><summary><b>Hint 1</b></summary>

Signal: 'penalty', 'sparse', 'shrink'. Pattern: regularization; the geometry of the constraint explains the zeros.

</details>

<details><summary><b>Hint 2</b></summary>

1. Ridge: `loss + lambda * sum(b^2)`; lasso: `loss + lambda * sum(|b|)`. 2. L2 shrinks smoothly, never exactly to 0. 3. L1 has a corner at 0, so coefficients hit exactly 0. 4. Scale features first.

</details>

<details><summary><b>Solution</b></summary>

**Model answer (about 90 s):**

"Both add a penalty on coefficient size to the training loss. Ridge minimises `sum (y - Xb)^2 + lambda * sum b_j^2`; lasso uses `lambda * sum |b_j|`. Lambda trades bias for variance: bigger lambda, simpler model.

Ridge shrinks all coefficients smoothly toward zero but never exactly to zero. It is great with many correlated features: it spreads the weight across them and stabilises the estimates, and it has a closed form, `(X^T X + lambda I)^-1 X^T y`.

Lasso sets some coefficients exactly to zero, so it does feature selection. Geometrically the L1 constraint is a diamond with corners on the axes, and the loss contours usually touch it at a corner. Its gradient does not vanish near zero, unlike the L2 penalty, whose pull fades as b approaches 0. Downside: with correlated features it picks one somewhat arbitrarily and is unstable.

Elastic net mixes both: sparse like lasso but keeps groups of correlated features like ridge. In all cases standardise the features first, because the penalty depends on their scale, and do not penalise the intercept. Choose lambda by cross-validation."

**Why:** The key words: penalty formulas, 'exactly zero' and why (corners of the L1 ball), correlated features, scaling, and choosing lambda by CV. The Bayesian view (L2 = Gaussian prior, L1 = Laplace prior) is a good bonus.

**Common mistakes:** Saying ridge does feature selection. Forgetting to scale. Choosing lambda on the test set.

**Learn more:** [ai-regularization-cv](https://mahsa-agz.github.io/ds-handbook/#ai-regularization-cv)

</details>

---
### Q2. Watch coefficients disappear

*easy, about 6 min*

Use the diabetes data (10 features). Standardise the features, then for `alpha` in 0.1, 1, 5, 10, 30 fit `Lasso(alpha)` and `Ridge(alpha)` on all rows and print how many coefficients are non-zero for each.

Then fit `LassoCV(cv=5, random_state=0)` (after scaling) and print the chosen alpha and its number of non-zero coefficients. Which features would you report to a doctor as 'not needed'?

In [ ]:
from sklearn.datasets import load_diabetes

# your code here

<details><summary><b>Hint 1</b></summary>

Signal: 'which features can we drop?'. Pattern: L1 regularization path, alpha chosen by CV.

</details>

<details><summary><b>Hint 2</b></summary>

1. `make_pipeline(StandardScaler(), Lasso(alpha=a))`. 2. The fitted model is the last step: `pipe[-1].coef_`. 3. Count `coef_ != 0`. 4. `LassoCV` tries a grid of alphas and keeps the best CV error.

</details>

<details><summary><b>Solution</b></summary>

```python
from sklearn.datasets import load_diabetes
from sklearn.linear_model import Lasso, LassoCV, Ridge
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

X, y = load_diabetes(return_X_y=True)
names = load_diabetes().feature_names
for alpha in [0.1, 1, 5, 10, 30]:
    lasso = make_pipeline(StandardScaler(), Lasso(alpha=alpha)).fit(X, y)
    ridge = make_pipeline(StandardScaler(), Ridge(alpha=alpha)).fit(X, y)
    nz_l = int((lasso[-1].coef_ != 0).sum())
    nz_r = int((ridge[-1].coef_ != 0).sum())
    print(f"alpha {alpha:>4}: lasso non-zero {nz_l:2d}   ridge non-zero {nz_r:2d}")
    if alpha == 10:
        print("   kept at alpha 10:", [n for n, c in zip(names, lasso[-1].coef_) if c != 0])
# out: alpha  0.1: lasso non-zero  9   ridge non-zero 10
# out: alpha    1: lasso non-zero  7   ridge non-zero 10
# out: alpha    5: lasso non-zero  5   ridge non-zero 10
# out: alpha   10: lasso non-zero  4   ridge non-zero 10
# out: kept at alpha 10: ['bmi', 'bp', 's3', 's5']
# out: alpha   30: lasso non-zero  2   ridge non-zero 10

cv_model = make_pipeline(StandardScaler(), LassoCV(cv=5, random_state=0)).fit(X, y)
print("LassoCV alpha:", round(cv_model[-1].alpha_, 3), " non-zero:", int((cv_model[-1].coef_ != 0).sum()))
print("dropped by LassoCV:", [n for n, c in zip(names, cv_model[-1].coef_) if c == 0])
# out: LassoCV alpha: 0.079  non-zero: 9
# out: dropped by LassoCV: ['s3']
```

**Why:** Ridge keeps all 10 coefficients non-zero at every alpha; it only shrinks them. Lasso drops features as alpha grows: 9, 7, 5, 4, then 2 survive. CV picks a small alpha (0.079) that keeps 9 of 10, so for prediction almost every feature helps a little. The honest answer to the doctor: only `s3` is dropped, and even that is fragile. At alpha 10, `s3` is one of the 4 survivors (with bmi, bp and s5), but at the CV alpha it is the one dropped: s1 to s6 are correlated blood measures, so lasso swaps between them. 'Not selected' means 'not needed given the others', not 'no effect'. To report stable selections, bootstrap the lasso and count how often each feature survives.

**Complexity:** Coordinate descent on 442 x 10: milliseconds per alpha; LassoCV fits 100 alphas x 5 folds in well under a second.

**Common mistakes:** Not scaling (the penalty then depends on the units). Reading 'lasso dropped it' as 'it has no effect': with correlated features (here the six blood-serum measures) lasso keeps one and drops the others. In sklearn, `alpha` is the penalty strength (bigger = simpler), while in `LogisticRegression` the knob is `C = 1 / lambda` (bigger = less regularization).

**Learn more:** [ai-regularization-cv](https://mahsa-agz.github.io/ds-handbook/#ai-regularization-cv), [ai-linear-regression](https://mahsa-agz.github.io/ds-handbook/#ai-linear-regression)

</details>

---
### Q3. Eighty percent on random labels

*easy, about 7 min*

This question uses **simulated** data on purpose: 100 rows, 2,000 features of pure noise, and random 0/1 labels (`np.random.default_rng(0)`, normal features, then integer labels). No model can beat 50% on new data.

1. The wrong way: select the 20 features most related to y (`SelectKBest(f_classif, k=20)`) on all rows, then run 5-fold CV of a logistic regression on those 20 features.
2. The right way: put the selection inside a pipeline and cross-validate the pipeline.

Use `StratifiedKFold(5, shuffle=True, random_state=0)` and print both CV accuracies. Explain the gap.

*Source: [Hastie, Tibshirani, Friedman, The Elements of Statistical Learning, section 7.10.2](https://hastie.su.domains/ElemStatLearn/)*

In [ ]:
from sklearn.feature_selection import SelectKBest, f_classif
rng = np.random.default_rng(0)
X = rng.normal(size=(100, 2000))
y = rng.integers(0, 2, size=100)

# your code here

<details><summary><b>Hint 1</b></summary>

Signal: a data-dependent preprocessing step done before the split. Pattern: leakage in cross-validation; every fitted step must live inside the CV loop.

</details>

<details><summary><b>Hint 2</b></summary>

1. Wrong: `X_sel = SelectKBest(...).fit_transform(X, y)`, then `cross_val_score(lr, X_sel, y, cv=cv)`. 2. Right: `make_pipeline(SelectKBest(...), LogisticRegression(...))` and `cross_val_score(pipe, X, y, cv=cv)`.

</details>

<details><summary><b>Solution</b></summary>

```python
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.pipeline import make_pipeline

rng = np.random.default_rng(0)
X = rng.normal(size=(100, 2000))         # 2,000 pure-noise features
y = rng.integers(0, 2, size=100)         # random labels: true accuracy is 50%
cv = StratifiedKFold(5, shuffle=True, random_state=0)

# WRONG: pick the 20 "best" features using ALL rows, then cross-validate
X_sel = SelectKBest(f_classif, k=20).fit_transform(X, y)
wrong = cross_val_score(LogisticRegression(max_iter=1000), X_sel, y, cv=cv).mean()

# RIGHT: selection is a step of the pipeline, refit inside each training fold
pipe = make_pipeline(SelectKBest(f_classif, k=20), LogisticRegression(max_iter=1000))
right = cross_val_score(pipe, X, y, cv=cv).mean()
print("selection outside CV:", round(wrong, 2))
print("selection inside CV: ", round(right, 2))
# out: selection outside CV: 0.81
# out: selection inside CV:  0.52
```

**Why:** Selecting on all 100 rows lets the labels of the validation folds choose the features. Among 2,000 noise columns, some correlate with these particular random labels by chance, and the wrong CV then rewards that chance fit: 0.81 accuracy on labels that are pure coin flips. Inside a pipeline the selection is refit on each training fold only, and the honest estimate is 0.52, about chance. Rule: every step that looks at the data (scaling, imputation, selection, encoding, resampling) is part of the model and must be fitted inside the CV loop.

**Complexity:** 20 tiny logistic fits plus 5 F-tests over 2,000 columns: under a second.

**Common mistakes:** Scaling, imputing, target-encoding, oversampling (SMOTE) or selecting features on the full data before CV: all are the same leak. Tuning hyperparameters on the same folds you report (use nested CV or a final untouched test set).

**Learn more:** [ai-regularization-cv](https://mahsa-agz.github.io/ds-handbook/#ai-regularization-cv), [ai-features-imbalance](https://mahsa-agz.github.io/ds-handbook/#ai-features-imbalance)

</details>

---
### Q4. Pick C without touching the test set

*easy, about 5 min*

Use the breast cancer data. Hold out 25% as a test set (stratified, `random_state=0`). On the training part, tune `C` of a scaled logistic regression over `[0.001, 0.01, 0.1, 1, 10, 100]` with 5-fold stratified CV (`shuffle=True, random_state=0`) and ROC AUC as the score.

Print the CV AUC for each C, the best C, and the test AUC of the refitted best model. Which side of the grid is underfitting?

In [ ]:
from sklearn.datasets import load_breast_cancer

# your code here

<details><summary><b>Hint 1</b></summary>

Signal: a hyperparameter plus a held-out test set. Pattern: GridSearchCV on the training data only, then one final test evaluation.

</details>

<details><summary><b>Hint 2</b></summary>

1. Pipeline(StandardScaler, LogisticRegression). 2. Param name: `logisticregression__C`. 3. `GridSearchCV(..., scoring='roc_auc')`, fit on train. 4. `grid.predict_proba(X_te)` uses the refitted best model.

</details>

<details><summary><b>Solution</b></summary>

```python
from sklearn.datasets import load_breast_cancer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import GridSearchCV, StratifiedKFold, train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

X, y = load_breast_cancer(return_X_y=True)
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.25, stratify=y, random_state=0)
pipe = make_pipeline(StandardScaler(), LogisticRegression(max_iter=5000))
grid = GridSearchCV(pipe, {"logisticregression__C": [0.001, 0.01, 0.1, 1, 10, 100]},
                    cv=StratifiedKFold(5, shuffle=True, random_state=0), scoring="roc_auc")
grid.fit(X_tr, y_tr)
for C, s in zip(grid.cv_results_["param_logisticregression__C"], grid.cv_results_["mean_test_score"]):
    print(f"C {C:>7}: CV AUC {s:.4f}")
print("best C:", grid.best_params_["logisticregression__C"])
print("test AUC:", round(roc_auc_score(y_te, grid.predict_proba(X_te)[:, 1]), 4))
# out: C   0.001: CV AUC 0.9892
# out: C    0.01: CV AUC 0.9933
# out: C     0.1: CV AUC 0.9953
# out: C     1.0: CV AUC 0.9947
# out: C    10.0: CV AUC 0.9902
# out: C   100.0: CV AUC 0.9876
# out: best C: 0.1
# out: test AUC: 0.9929
```

**Why:** The CV AUC peaks at C = 0.1 (0.9953). Small C means strong regularization: C = 0.001 underfits (0.9892). Large C means weak regularization: C = 100 overfits a little (0.9876). The curve is flat near the top (0.1 and 1 differ by 0.0006), so any C from 0.01 to 1 is fine. The test AUC of the refitted model is 0.9929, close to the CV estimate, which suggests no leakage. Stratified folds keep the benign/malignant ratio the same in every fold, which makes the fold scores less noisy.

**Complexity:** 6 values x 5 folds = 30 fits plus one refit, each milliseconds.

**Common mistakes:** Putting the scaler outside the pipeline (fitted on all training folds, a small leak). Reading `C` like lasso's alpha (in LogisticRegression a small C means strong regularization). Reporting the best CV score as the expected performance: it is optimistically biased, which is why we keep a test set.

**Learn more:** [ai-regularization-cv](https://mahsa-agz.github.io/ds-handbook/#ai-regularization-cv), [ai-classification-metrics](https://mahsa-agz.github.io/ds-handbook/#ai-classification-metrics)

</details>

---
### Q5. Turn the penalty knob (review)

*easy, about 3 min*

You increase the ridge penalty lambda from 0 to a huge value. Describe what happens to the training error, the validation error, the bias and the variance along the way. Where is the best lambda, and how do you find it without using the test set?

*Your answer (double-click to write):*

<details><summary><b>Hint 1</b></summary>

Signal: one knob that controls complexity. Pattern: bias-variance trade-off (regularization path).

</details>

<details><summary><b>Hint 2</b></summary>

1. lambda = 0: plain least squares (lowest bias, highest variance). 2. lambda huge: all coefficients near 0, the model predicts the mean. 3. Validation error is U-shaped; pick its minimum with K-fold CV on the training data.

</details>

<details><summary><b>Solution</b></summary>

**Model answer:** "At lambda = 0 we have ordinary least squares: lowest training error, lowest bias, highest variance. As lambda grows, the coefficients shrink, so training error rises steadily (the model is constrained), bias rises and variance falls. Validation error first falls, because the variance we remove is worth more than the bias we add, then rises once the model underfits. At a huge lambda all slopes are near zero and the model predicts the mean: high bias, almost zero variance.

The best lambda is at the bottom of the validation curve. Find it with K-fold cross-validation on the training data over a log-spaced grid (for example 1e-3 to 1e3), optionally with the one-standard-error rule to prefer the simpler model. The test set is used once at the end."

**Why:** It connects regularization to the bias-variance picture you learned on day 2: lambda is a complexity knob, just like tree depth or k in kNN.

**Common mistakes:** Saying training error can go down as lambda grows. Using a linear grid instead of a log grid. Picking lambda on the test set.

**Learn more:** [ai-bias-variance](https://mahsa-agz.github.io/ds-handbook/#ai-bias-variance), [ai-regularization-cv](https://mahsa-agz.github.io/ds-handbook/#ai-regularization-cv)

</details>

---
## Redo list

Write down every question you got wrong, solved only after Hint 2, or finished over time. Solve it again from scratch 3 days and 7 days later.

**Done for today.** Finish with the [theory flashcards for day 6](https://mahsa-agz.github.io/ds-handbook/#day-6).